In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'AB-тест'
F = 'Arial'

headers = ['Сравнение','n контр.','n тест','Ср. контр.','Ср. тест','Разница',
           'Uplift %','ДИ 2.5%','ДИ 97.5%','p-value','Значимость']
ncol = len(headers)

ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=ncol)
ws.cell(1,1,'Результаты AB-теста (Poisson bootstrap): средняя сумма транзакции').font = Font(name=F, bold=True, size=14)

thin = Side(style='thin', color='B7B7B7')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for c, h in enumerate(headers, 1):
    cell = ws.cell(3, c, h)
    cell.font = Font(name=F, bold=True, color='FFFFFF')
    cell.fill = PatternFill('solid', fgColor='2F5496')
    cell.alignment = Alignment(horizontal='center', wrap_text=True)
    cell.border = border

sig_fill = PatternFill('solid', fgColor='C6EFCE')
nosig_fill = PatternFill('solid', fgColor='FCE4D6')

for i, row in results_df.iterrows():
    x_c = df_agg.loc[df_agg['type_group']==row['control'], 'bic_ruramnt'].dropna().to_numpy(float)
    x_t = df_agg.loc[df_agg['type_group']==row['test'], 'bic_ruramnt'].dropna().to_numpy(float)
    r = 4 + i
    mean_c, mean_t = float(x_c.mean()), float(x_t.mean())
    obs_diff, p_val = float(row['obs_diff']), float(row['p_value'])
    uplift = obs_diff/mean_c if mean_c else float('nan')
    is_sig = p_val < 0.05
    vals = [f"{row['test']} vs {row['control']}", len(x_c), len(x_t),
            round(mean_c,2), round(mean_t,2), round(obs_diff,2),
            round(uplift,4), round(float(row['ci_low']),2), round(float(row['ci_high']),2),
            round(p_val,4), 'Значимо' if is_sig else 'Не значимо']
    for c, v in enumerate(vals, 1):
        ws.cell(r, c, v)
    fill = sig_fill if is_sig else nosig_fill
    fmt = {4:'0.00',5:'0.00',6:'0.00',7:'0.0%',8:'0.00',9:'0.00',10:'0.0000'}
    for c in range(1, ncol+1):
        cell = ws.cell(r,c)
        cell.font = Font(name=F)
        cell.border = border
        cell.fill = fill
        if c in fmt:
            cell.number_format = fmt[c]

for c, w in enumerate([26,9,8,11,10,10,9,10,10,10,13], 1):
    ws.column_dimensions[get_column_letter(c)].width = w
ws.freeze_panes = 'A4'

wb.save('ab_test_bootstrap_report.xlsx')
print('Готово: ab_test_bootstrap_report.xlsx')